# Manual curation in the SpikeInterface GUI (Spyglass pipeline, step 3 of 4)

This notebook hand-curates the sorting produced by the Spyglass pipeline using the
[SpikeInterface GUI](https://github.com/SpikeInterface/spikeinterface-gui). It is the one step in
the chain that does **not** run in the `spyglass` environment.

> **Environment:** run this notebook with the **`spikeinterface_gui_env`** kernel
> (SpikeInterface 0.104 + `spikeinterface-gui`). The `spyglass` environment has SpikeInterface 0.99
> and no GUI, so it cannot open the analyzer or launch the GUI.

**Inputs:** the `recording` and `sorting` folders exported by
[`Pipeline_Spyglass_AutomaticCuration.ipynb`](Pipeline_Spyglass_AutomaticCuration.ipynb).
**Output:** a `curation_data.json` file that
[`Pipeline_Spyglass_CompareCurations.ipynb`](Pipeline_Spyglass_CompareCurations.ipynb) re-ingests
back into Spyglass.

In [1]:
from pathlib import Path

import spikeinterface as si
from spikeinterface_gui import run_mainwindow

/opt/anaconda3/envs/spikeinterface_gui_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Point at the exported sorting

Set `export_dir` to the path printed by `Pipeline_Spyglass_AutomaticCuration.ipynb` (it ends in the
`sorting_id`).

In [2]:
# Paste the sorting_id printed by Pipeline_Spyglass_AutomaticCuration.ipynb:
sorting_id = "62224b1f-3d21-498f-bdc3-02c7e398ce77"

# Same handoff folder as Pipeline_Spyglass_AutomaticCuration.ipynb's export_root (kept outside the
# git repo). Must match the export_root set there.
export_dir = Path("/Users/pauladkisson/Documents/CatalystNeuro/DudchenkoConv/curation_exports") / sorting_id

assert (export_dir / "recording").exists() and (export_dir / "sorting").exists(), (
    f"Exported recording/sorting not found under {export_dir}. "
    "Run Pipeline_Spyglass_AutomaticCuration.ipynb (step 2) first and copy the path it prints."
)

## Load the recording and sorting

These were written by SpikeInterface 0.99 in the `spyglass` environment; here we read them with
SpikeInterface 0.104. The npz sorting and binary recording formats are portable across these
versions (you may see a version-mismatch warning, which is safe to ignore).

We also project the recording's probe to 2D — Spyglass stores a 3D probe (with `z = 0`), and
SpikeInterface's `unit_locations` estimator assumes 2D contacts and would otherwise crash. The
projection is lossless here since `z` is constant.

In [3]:
recording = si.load(export_dir / "recording")
sorting = si.load(export_dir / "sorting")

# Spyglass attaches a 3D probe (the contacts lie in the xy-plane with z = 0). SpikeInterface's
# unit-location estimator (`monopolar_triangulation`) assumes 2D contacts and crashes on a 3D
# probe, so project to the xy-plane. This is lossless here because every contact has z = 0.
recording = recording.set_probe(recording.get_probe().to_2d())

print(recording)
print(sorting)

BinaryFolderRecording: 32 channels - 30000.000000 Hz - 1 segments - 36,580,095 samples 
                       1,219.34s (20.32 minutes) - float64 dtype - 8.72 GiB
NumpyFolder (NumpyFolderSorting): 19 units - 1 segments - 30.0kHz


## Build a SortingAnalyzer

The GUI is driven by a `SortingAnalyzer` plus a set of computed extensions (waveforms, templates,
amplitudes, correlograms, quality metrics, ...). We build it once and save it next to the export so
the GUI can write its curation file alongside.

In [4]:
analyzer = si.create_sorting_analyzer(
    sorting,
    recording,
    folder=export_dir / "sorting_analyzer",
    format="binary_folder",
    overwrite=True,
)
analyzer.compute(
    [
        "random_spikes",
        "waveforms",
        "templates",
        "noise_levels",
        "spike_amplitudes",
        "correlograms",
        "unit_locations",
        "template_similarity",
    ]
)
analyzer.compute("quality_metrics")

noise_level (no parallelization): 100%|██████████| 20/20 [00:00<00:00, 87.64it/s]
/opt/anaconda3/envs/spikeinterface_gui_env/lib/python3.13/site-packages/spikeinterface/postprocessing/template_similarity.py:345: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  overlapping_ids = overlapping_j_list[i]
Compute : spike_amplitudes (no parallelization): 100%|██████████| 1220/1220 [00:04<00:00, 284.58it/s]
/opt/anaconda3/envs/spikeinterface_gui_env/lib/python3.13/site-packages/spikeinterface/core/analyzer_extension_core.py:1165: UserWarning: The following metrics will not be computed due to missing dependencies: ['d_prime', 'nearest_neighbor', 'silhouette', 'drift', 'mahalanobis']
  warnings.warn(
/opt/anaconda3/envs/spikeinterface_gui_env/lib/python3.13/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/envs/spikeinterface_gui_env/lib/python3.

## Launch the GUI and curate

Running the next cell opens the desktop GUI with curation enabled. In the GUI you can:

- **merge** units that were over-split (select them, then merge),
- **label** units (`good` / `noise` / `MUA`), and
- **remove** units that are noise.

When you are done, **save in analyzer** from within the GUI (it writes
`sorting_analyzer/spikeinterface_gui/curation_data.json` inside `export_dir`). Then move on to
`Pipeline_Spyglass_CompareCurations.ipynb`.

> **Terminal alternative.** Instead of this cell you can launch the same GUI from a shell:
> ```
> conda run -n spikeinterface_gui_env sigui <export_dir>/sorting_analyzer --curation --mode desktop
> ```

In [ ]:
run_mainwindow(analyzer, mode="desktop", curation=True)

<spikeinterface_gui.backend_qt.QtMainWindow(0x33dfaee10) at 0x33e5b2100>

: 

## Done

The GUI saved your curation to:

```
<export_dir>/sorting_analyzer/spikeinterface_gui/curation_data.json
```

Switch back to the **`spyglass`** kernel and run
[`Pipeline_Spyglass_CompareCurations.ipynb`](Pipeline_Spyglass_CompareCurations.ipynb) to ingest it
and compare against the raw and automatic curations.